In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('../retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [4]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('../selic/selic_diario.csv').set_index('date')

In [5]:
selic_d

,valor_anual,valor_diario
date,,
2026-07-23,14.25,0.01087
2026-07-22,14.25,0.01087
2026-07-21,14.25,0.01087
2026-07-20,14.25,0.01087
2026-07-19,14.25,0.01087
...,...,...
2016-01-05,14.25,0.01087
2016-01-04,14.25,0.01087
2016-01-03,14.25,0.01087


In [6]:
##### EXCESSO PARA TODOS
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]

    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))

        print(f"============\n SIGMA {ano}\n===========")            

    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2025
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 SIGMA 2024
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 SIGMA 2023
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 SIGMA 2022
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2021
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 SIGMA 2020
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2019
 EXCESSO 2018
Atualização, Ano:  2018
Tamanho DF de 2018:  119
Tamanho Selic:  119
tamanho final do Excesso:  119
 SIG

In [7]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

## MODEO FINAL COM TODOS OS ANOS

In [34]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        #SCORE MF------------
        score_todos_anos = pd.read_csv(f'../score_mf/{ano}/mf_{ano}.csv').set_index('ano')
        score_usado = score_todos_anos.dropna(axis=1).copy()

        score_min = score_usado.min().min()
        score_max = score_usado.max().max()
        media_score = np.array(score_usado).mean()
        # print(score_usado)
        print("Score Atualizado")
        print("Score Min:",score_min)
        print("Score MAX:",score_max)

        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado[score_usado.columns].copy()

        # COV
        df_cov = retorno_usado.cov()
        print("VARIANCIA Atualizado")
        
        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()


    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
    model.sigma_min = pyo.Param(initialize=0, mutable=True)
    model.sigma_max = pyo.Param(initialize=1, mutable=True)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.theta = pyo.Param(initialize=vb_theta)
    # model.score_min = pyo.Param(initialize=score_min)
    # model.score_max = pyo.Param(initialize=score_max)
    model.score_m = pyo.Param(initialize= media_score)
    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.KMAX = pyo.Param(initialize=100)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
    # CARDINALIDADE
    # era sum(model.y[a] ...) >= e <=

    # Com y=0
    def res_y_zero(model,a):
        return model.w[a] <= model.KMAX * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

    # com y=1
        #  segundo teto
    def res_y_1_teto2(model,a):
        return model.w[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

        # Limite inf 1
    def limite_1(model,a):
        return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)

    #  PESOS MIN MAX
    def pesomin(model,a):
        return model.z[a] >= vb_peso_minimo * model.w[a]
    model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
        return model.z[a] <= vb_peso_maximo * model.w[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    #  cardinalidade
    def card_min(model):
        return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
        return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
        return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
        return sum(sum(model.retornos_ativos[t,a]*model.z[a] for a in model.ativos) for t in model.dias) == 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)
    
    def score_r(model):
        return sum(model.score[a]*model.y[a] for a in model.ativos) >= vb_cardinalidade_max * model.score_m
    model.r_score = pyo.Constraint(rule=score_r)
    ## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
    def obj_1(model):
        var = sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)
        # var_normalisum(model.y[a] * model.score[a] for a in model.ativos)
        return var
    model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.minimize)
    # ------------------- solver 1
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=True,symbolic_solver_labels=True)
    model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
    # if res.opt.termination_condition  == pyo.TerminationCondition.optimal:
    #     print(f"-----MODELO FOI RESOLVIDO {an}------------")
    # else:
    #     print(f"MODEO NAO FOI RESOLVIDO {an}----------")
    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[ano_int+1] = {
        'pesos':  melhor_pesos,
        'z[a]': {a:model.z[a].value for a in model.ativos},
        'score': {a:model.score[a] for a in model.ativos}
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }

    print(f"{ano} -> {melhor_pesos}")
    print("+-="*30)
    print("=====================")
    print(model.display())
    print("=====================")
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2025
Modelo Antigo deletado 
 Iniciando Novo
Score Atualizado
Score Min: 0.0
Score MAX: 1.0
VARIANCIA Atualizado
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2026
## UTILIZANDO DADOS DE RETORNO DE:  2025

Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmp_cy45p4f

In [37]:
carteiras_anuais[2016]


{'pesos': {'ABEV3': 0.0,
  'ANIM3': 0.0,
  'AXIA3': 0.0,
  'AZZA3': 0.0,
  'B3SA3': 0.0,
  'BBSE3': 0.0,
  'BEEF3': 0.0,
  'BRAP4': 0.0,
  'BRKM5': 0.12453372943246492,
  'CMIG4': 0.0,
  'COGN3': 0.0,
  'CPFE3': 0.0,
  'CPLE3': 0.0,
  'CSAN3': 0.09608144534693083,
  'CSMG3': 0.0,
  'CSNA3': 0.0,
  'CVCB3': 0.0,
  'CYRE3': 0.0,
  'DIRR3': 0.04292844392062733,
  'ECOR3': 0.0,
  'EMBJ3': 0.0,
  'ENGI11': 0.05740466746120151,
  'EQTL3': 0.0,
  'EZTC3': 0.0,
  'FLRY3': 0.14873285366897226,
  'GGBR4': 0.0,
  'GOAU4': 0.0,
  'HYPE3': 0.19999999999999998,
  'ISAE4': 0.0,
  'ITSA4': 0.0,
  'JHSF3': 0.0,
  'KLBN11': 0.0,
  'LREN3': 0.0,
  'MGLU3': 0.03349677523165688,
  'MOTV3': 0.0,
  'MRVE3': 0.20000000000000004,
  'MULT3': 0.0,
  'PETR3': 0.0,
  'PETR4': 0.0,
  'POMO4': 0.02,
  'PRIO3': 0.0,
  'PSSA3': 0.0,
  'RADL3': 0.0,
  'RAPT4': 0.0,
  'RENT3': 0.0,
  'SBSP3': 0.07682208493814624,
  'SLCE3': 0.0,
  'SUZB3': 0.0,
  'TAEE11': 0.0,
  'TOTS3': 0.0,
  'UGPA3': 0.0,
  'USIM5': 0.0,
  'VALE3': 

In [ ]:
print(max(carteiras_anuais[2018]['z[a]'].values()))
print(min(carteiras_anuais[2018]['z[a]'].values()))


0.4723832562736914
0.0


In [39]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        # print(v)
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
2016


In [43]:
df_portfolios[df_portfolios['ano']==2024]

,ano,ativo,peso
19,2024,CSMG3,0.0466
20,2024,CXSE3,0.2000
21,2024,EMBJ3,0.1997
22,2024,PETR4,0.0661
23,2024,POMO4,0.0211
24,2024,PSSA3,0.0200
25,2024,SBSP3,0.0342
26,2024,SUZB3,0.1249
27,2024,UGPA3,0.1463
28,2024,USIM5,0.1410


In [ ]:
df_portfolios.to_csv('carteiras_mf_linear_sharpe.csv')
